# 02 — Baseline Sequential RAG Evaluation
Build FAISS index, evaluate 200 questions per dataset, record T_sequential.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

In [2]:
import json, time
import pandas as pd
from config import ACCURACY_DIR, INDEX_DIR, LOG_DIR, CORPUS_DIR

os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

## Build Sequential Index (corpus_1000) + Record T_sequential

In [3]:
from pipeline.chunker import chunk_documents
from pipeline.embedder import embed_chunks, get_model
from pipeline.indexer import build_index, save_index

with open(os.path.join(CORPUS_DIR, 'corpus_1000.json')) as f:
    corpus = json.load(f)

t_start = time.perf_counter()
chunks     = chunk_documents(corpus)
embeddings = embed_chunks(chunks)
index      = build_index(embeddings)
t_seq      = time.perf_counter() - t_start

print(f'T_sequential (1k docs, {len(chunks)} chunks): {t_seq:.2f}s')

# save
index_prefix = os.path.join(INDEX_DIR, 'baseline_1k')
save_index(index, chunks, index_prefix)

# log T_sequential
latency_log = {'corpus_size': 1000, 'n_chunks': len(chunks), 'T_sequential_s': t_seq}
with open(os.path.join(LOG_DIR, 'T_sequential.json'), 'w') as f:
    json.dump(latency_log, f)
print('Index and latency log saved.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

T_sequential (1k docs, 2658 chunks): 96.81s
Index and latency log saved.


## LLM Setup (local Ollama)

In [4]:
from config import OLLAMA_BASE_URL, OLLAMA_MODEL

# Sanity check: Ollama server reachable
from openai import OpenAI as _OpenAI
_client = _OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
_resp = _client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": "Say OK"}],
)
print(f'Ollama check ({OLLAMA_MODEL}): {_resp.choices[0].message.content.strip()}')

from pipeline.generator import GeminiGenerator
generator = GeminiGenerator()

Ollama check (qwen2.5:3b): OK


## Evaluate on FinQA (Exact Match)

In [5]:
from pipeline.baseline import run_baseline
from pipeline.embedder import get_model
from evaluation.metrics import evaluate_dataset

embed_model = get_model()

with open(os.path.join(ACCURACY_DIR, 'finqa_eval_questions.json')) as f:
    finqa_eval = json.load(f)

def baseline_fn(q):
    return run_baseline(q, index, chunks, embed_model, generator)

finqa_results = evaluate_dataset(
    finqa_eval,
    baseline_fn,
    log_path=os.path.join(ACCURACY_DIR, 'baseline_finqa.jsonl')
)
print('FinQA Baseline:', finqa_results)

[checkpoint] resuming — 11 done, 189 remaining
  [20/200] em=0.000  f1=0.001  errors=0
  [30/200] em=0.000  f1=0.002  errors=0
  [40/200] em=0.000  f1=0.002  errors=0
  [50/200] em=0.000  f1=0.001  errors=0
  [60/200] em=0.000  f1=0.002  errors=0
  [70/200] em=0.000  f1=0.001  errors=0
  [80/200] em=0.000  f1=0.001  errors=0
  [90/200] em=0.000  f1=0.001  errors=0
  [100/200] em=0.000  f1=0.001  errors=0
  [110/200] em=0.000  f1=0.001  errors=0
  [120/200] em=0.000  f1=0.001  errors=0
  [130/200] em=0.000  f1=0.001  errors=0
  [140/200] em=0.000  f1=0.001  errors=0
  [150/200] em=0.000  f1=0.001  errors=0
  [160/200] em=0.000  f1=0.001  errors=0
  [170/200] em=0.000  f1=0.001  errors=0
  [180/200] em=0.000  f1=0.001  errors=0
  [190/200] em=0.000  f1=0.001  errors=0
  [200/200] em=0.000  f1=0.001  errors=0
FinQA Baseline: {'em': 0.0, 'f1': 0.0009860599338626706, 'avg_latency_ms': 84041.31877501012, 'n': 200, 'errors': 0}


## Evaluate on MultiHop-RAG (F1)

In [6]:
with open(os.path.join(ACCURACY_DIR, 'multihop_eval_questions.json')) as f:
    multihop_eval = json.load(f)

multihop_results = evaluate_dataset(
    multihop_eval,
    baseline_fn,
    log_path=os.path.join(ACCURACY_DIR, 'baseline_multihop.jsonl')
)
print('MultiHop Baseline:', multihop_results)

  [10/200] em=0.000  f1=0.000  errors=0
  [20/200] em=0.000  f1=0.006  errors=0
  [30/200] em=0.000  f1=0.006  errors=0
  [40/200] em=0.000  f1=0.007  errors=0
  [50/200] em=0.000  f1=0.008  errors=0
  [60/200] em=0.000  f1=0.007  errors=0
  [70/200] em=0.000  f1=0.006  errors=0
  [80/200] em=0.000  f1=0.007  errors=0
  [90/200] em=0.000  f1=0.007  errors=0
  [100/200] em=0.000  f1=0.007  errors=0
  [110/200] em=0.000  f1=0.007  errors=0
  [120/200] em=0.000  f1=0.008  errors=0
  [130/200] em=0.000  f1=0.008  errors=0
  [140/200] em=0.000  f1=0.008  errors=0
  [150/200] em=0.000  f1=0.008  errors=0
  [160/200] em=0.000  f1=0.008  errors=0
  [170/200] em=0.000  f1=0.008  errors=0
  [180/200] em=0.000  f1=0.008  errors=0
  [190/200] em=0.000  f1=0.008  errors=0
  [200/200] em=0.000  f1=0.008  errors=0
MultiHop Baseline: {'em': 0.0, 'f1': 0.007684360440577707, 'avg_latency_ms': 60424.1790240302, 'n': 200, 'errors': 0}


## Save Summary

In [7]:
summary = {
    'finqa_em':      finqa_results['em'],
    'multihop_f1':   multihop_results['f1'],
    'finqa_latency_ms':    finqa_results['avg_latency_ms'],
    'multihop_latency_ms': multihop_results['avg_latency_ms'],
}
with open(os.path.join(ACCURACY_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved baseline_results.json')
print(summary)

Saved baseline_results.json
{'finqa_em': 0.0, 'multihop_f1': 0.007684360440577707, 'finqa_latency_ms': 84041.31877501012, 'multihop_latency_ms': 60424.1790240302}
